# RuleShift Benchmark v1 — Kaggle Benchmark

Official `kaggle_benchmarks` notebook for the RuleShift Benchmark v1 cognitive flexibility benchmark.

- **Leaderboard task:** Binary (post-shift probe accuracy)
- **Companion task:** Narrative (same episodes, not leaderboard-scored)
- **Metric:** Post-shift Probe Accuracy = correct probes / total probes
- **Per-episode return:** `(num_correct, 4)`

In [ ]:
# Cell 1: Imports
from __future__ import annotations

import sys
from pathlib import Path

import kaggle_benchmarks as kbench

In [ ]:
# Cell 2: Repo path setup — resolve repo root and add src/ to sys.path
#
# On Kaggle, the repo is attached as a dataset under /kaggle/input/.
# The expected layout is:
#   /kaggle/input/<dataset-slug>/src/
#   /kaggle/input/<dataset-slug>/packaging/kaggle/frozen_artifacts_manifest.json
#
# Locally, the notebook lives at packaging/kaggle/ inside the repo root.

def _find_repo_root() -> Path:
    """Locate the repo root from the notebook's working directory or Kaggle input paths."""
    candidates: list[Path] = []
    seen: set[Path] = set()

    for origin in (Path.cwd().resolve(),):
        for candidate in (origin, *origin.parents):
            if candidate not in seen:
                seen.add(candidate)
                candidates.append(candidate)

    for search_root in (Path("/kaggle/input"), Path("/kaggle/working")):
        if not search_root.exists():
            continue
        for manifest_path in search_root.rglob("frozen_artifacts_manifest.json"):
            candidate = manifest_path.parents[2]
            if candidate not in seen:
                seen.add(candidate)
                candidates.append(candidate)

    for candidate in candidates:
        if (candidate / "src").is_dir() and (
            candidate / "packaging" / "kaggle" / "frozen_artifacts_manifest.json"
        ).is_file():
            return candidate

    raise FileNotFoundError(
        "Could not locate repo root. Expected src/ and "
        "packaging/kaggle/frozen_artifacts_manifest.json."
    )


REPO_ROOT = _find_repo_root()

if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

print(f"Repo root: {REPO_ROOT}")

In [ ]:
# Cell 3: Load frozen splits

from kaggle import BinaryResponse, normalize_binary_response, normalize_narrative_response, score_episode
from splits import PARTITIONS, load_frozen_split
from parser import PARSER_VERSION
from metrics import METRIC_VERSION
from render import render_binary_prompt, render_narrative_prompt

frozen_splits = {
    partition: load_frozen_split(partition)
    for partition in PARTITIONS
}

print(f"Parser version: {PARSER_VERSION}")
print(f"Metric version: {METRIC_VERSION}")
for partition in PARTITIONS:
    print(f"  {partition}: {len(frozen_splits[partition])} episodes")

In [ ]:
# Cell 4: Build evaluation dataframe — one row per (split, episode)

import pandas as pd

eval_rows: list[dict] = []
for partition in PARTITIONS:
    for record in frozen_splits[partition]:
        ep = record.episode
        eval_rows.append({
            "episode_id": ep.episode_id,
            "split": partition,
            "difficulty": ep.difficulty.value,
            "template_id": ep.template_id.value,
            "prompt_binary": render_binary_prompt(ep),
            "prompt_narrative": render_narrative_prompt(ep),
            "probe_targets": tuple(t.value for t in ep.probe_targets),
        })

eval_df = pd.DataFrame(eval_rows)
print(f"Evaluation rows: {len(eval_df)}")
eval_df[["episode_id", "split", "difficulty", "template_id"]].head()

In [ ]:
# Cell 5: Package-owned benchmark helpers
#
# The structured response schema, parser adapters, benchmark constants,
# and per-episode scoring logic are defined in src/kaggle.py.


In [ ]:
# Cell 6: Notebook-owned logic stays minimal
#
# This notebook only assembles frozen evaluation rows and wires kbench tasks
# to package-owned prompt, parser, and scoring helpers.


In [ ]:
# Cell 7: Task cells below remain submission-oriented.


In [ ]:
# Cell 8: @kbench.task — Binary (leaderboard-primary)
#
# The official Kaggle benchmark task for RuleShift Benchmark v1.
# Evaluates over frozen episodes. Returns (num_correct, 4) per episode.
#
# The function signature matches eval_df column names:
#   llm           — injected by kbench at evaluation time
#   prompt_binary — Binary prompt string for this episode
#   probe_targets — tuple of 4 frozen target label strings

@kbench.task(
    name="ruleshift_benchmark_v1_binary",
    description=(
        "Cognitive flexibility benchmark: infer a hidden rule shift from "
        "sparse contradictory evidence in a sequence of charge interactions, "
        "then predict four post-shift probe outcomes."
    ),
)
def ruleshift_benchmark_v1_binary(
    llm,
    prompt_binary: str,
    probe_targets: tuple,
) -> tuple[int, int]:
    try:
        response = llm.prompt(
            prompt_binary,
            schema=BinaryResponse,
        )
        predictions = normalize_binary_response(response)
    except Exception:
        predictions = None

    return score_episode(predictions, probe_targets)


# Run Binary evaluation over the frozen eval dataframe.
binary_results = ruleshift_benchmark_v1_binary.evaluate(
    llm=[kbench.llm],
    evaluation_data=eval_df,
)

In [ ]:
# Cell 9: @kbench.task — Narrative (companion, not leaderboard-scored)
#
# Same frozen episodes and probe targets as Binary.
# Uses the Narrative prompt format. Only the final four labels are scored.
# This task is companion evidence and must NOT be selected via %choose.
#
# The function signature matches eval_df column names:
#   llm              — injected by kbench at evaluation time
#   prompt_narrative — Narrative prompt string for this episode
#   probe_targets    — tuple of 4 frozen target label strings

@kbench.task(
    name="ruleshift_benchmark_v1_narrative",
    description=(
        "Narrative companion for RuleShift Benchmark v1. Same episodes as Binary, "
        "natural-language prompt format. Robustness evidence only — not leaderboard-scored."
    ),
)
def ruleshift_benchmark_v1_narrative(
    llm,
    prompt_narrative: str,
    probe_targets: tuple,
) -> tuple[int, int]:
    try:
        response = llm.prompt(prompt_narrative)
        predictions = normalize_narrative_response(response)
    except Exception:
        predictions = None

    return score_episode(predictions, probe_targets)


# Run Narrative evaluation over the same frozen eval dataframe.
# Non-blocking: Narrative is companion evidence only and does not affect the leaderboard score.
try:
    narrative_results = ruleshift_benchmark_v1_narrative.evaluate(
        llm=[kbench.llm],
        evaluation_data=eval_df,
    )
except Exception as _narrative_exc:
    # Narrative failure must not block the Binary leaderboard result.
    print(f"Narrative evaluation skipped: {_narrative_exc}")
    narrative_results = None

In [ ]:
# Cell 10: Result inspection (debug only — does not affect benchmark semantics)
#
# Displays the Binary evaluation result dataframe if available.
# This is for local verification only; Kaggle aggregates scores from the task return values.

try:
    binary_df = binary_results.as_dataframe()
    print(f"Binary evaluation rows: {len(binary_df)}")
    display(binary_df.head())
except Exception as _inspect_exc:
    print(f"Result inspection unavailable: {_inspect_exc}")

In [ ]:
# Cell 11: Local debug — dry-run scoring checks (does not run on Kaggle evaluation path)
#
# Verifies that scoring helpers produce correct outputs for known inputs.
# Safe to run locally before submitting; does not change benchmark semantics.

sample_row = eval_df.iloc[0]
print(f"Episode: {sample_row['episode_id']}")
print(f"Split: {sample_row['split']}")
print(f"Targets: {sample_row['probe_targets']}")
print()

# Verify scoring with a perfect prediction
perfect = score_episode(sample_row["probe_targets"], sample_row["probe_targets"])
print(f"Perfect prediction score: {perfect}")
assert perfect == (4, 4), f"Expected (4, 4), got {perfect}"

# Verify scoring with an invalid prediction
invalid = score_episode(None, sample_row["probe_targets"])
print(f"Invalid prediction score: {invalid}")
assert invalid == (0, 4), f"Expected (0, 4), got {invalid}"

# Verify normalize with a raw text response
targets_text = ", ".join(sample_row["probe_targets"])
normalized = normalize_binary_response(targets_text)
print(f"Parsed '{targets_text}' -> {normalized}")
assert normalized == sample_row["probe_targets"], f"Parse mismatch"

# Verify structured response path
labels = [Label(t) for t in sample_row["probe_targets"]]
structured = BinaryResponse(*labels)
structured_norm = normalize_binary_response(structured)
print(f"Structured response -> {structured_norm}")
assert structured_norm == sample_row["probe_targets"], f"Structured mismatch"

# Verify malformed text
malformed = normalize_binary_response("I don't know")
malformed_score = score_episode(malformed, sample_row["probe_targets"])
print(f"Malformed score: {malformed_score}")
assert malformed_score == (0, 4), f"Expected (0, 4), got {malformed_score}"

print("\nAll local checks passed.")

In [ ]:
# Cell 12: %choose — select Binary as the leaderboard task
%choose ruleshift_benchmark_v1_binary